In [1]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
from ytmusic_library import YTMusicPlaylists, PLAYLIST_TSV_COLUMNS


RUN_API_AUTH_TEST = True
HEADER_FILE='../headers_auth.json'
PLAYCOUNT_FILE='../playlists/_ytmusic_lastfm_match_id_map.tsv'
NOT_LIKE_PLAYLIST_TSV ='../playlists/_not_liked_tracks.tsv'
LIKE_PLAYLIST_TSV = '../playlists/_liked_tracks.tsv'
ALL_TRACKS_TSV = '../playlists/_tracks_db.tsv'
RADIO_TO_LIKE_PL_TSV='../playlists/_ytmusic_radio_to_like_pl_map.tsv'
BACKUP_DIR = '../playlists/'


In [2]:
Y = YTMusicPlaylists(
    header=HEADER_FILE, 
    playcount_map=PLAYCOUNT_FILE,  
    not_like_tsv=NOT_LIKE_PLAYLIST_TSV, 
    radio_to_like_map_tsv=RADIO_TO_LIKE_PL_TSV
)
if RUN_API_AUTH_TEST: Y.test_ytmusic_api()
print(f"Loaded {len(Y.playlists['title'].unique())} playlists")


Using header file: ../headers_auth.json
Loaded 266580 playounts from 104345 tracks
Test Passed in 2.80 seconds
Using ytmusicapi version: 0.25.0
Loaded 485 playlists


# Clean Up Radio Playlists

* Move LIKE to radios like playlist
* Remove DISLIKE and NOT LIKE

In [ ]:
playlist_names = [
    # 'x_r.WorldMusic_tracks_radio', # waaking caos
    # 'Soul Classic Sunshine radio',
    # 'rock instrumentals classic vintage radio', # moonchild ventures
    # 'jazz guitar radio', # golden earing
    # 'rock instrumentals classic vintage radio',
    # 'Soul Classic Sunshine radio'
    # 'electronic House Special radio', # show me
    'psychedelic classic rock radio',
    # 'electronic soft pad radio', # hatchet
    # 'Indie Dreams of Fall radio'
    
]
MIN_RADIO_LIKE_TO_SPLIT=3
for playlist_name in playlist_names:
    assert 'radio' in playlist_name
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId)
    clean_counters = Y.clean_up_playlist(
        pl_info, verbose=True, 
        move_like=True, min_num_like=MIN_RADIO_LIKE_TO_SPLIT,
        sleep=1, create_like_playlist=True, 
        remove_dislike=True, remove_not_like=True
    )


# Re-Sort Playlist based on LastFM playcount 

creates new sorted pl, deletes old


In [5]:
playlist_names = [
    'psychedelic classic rock radio'
]
USE_CACHE=False
for playlist_name in playlist_names:
    pl_info = Y.playlist_get_info(Y.query_by_title(playlist_name).playlistId, use_cache=USE_CACHE)
    pc_df = Y.playcount_sort_playlist(pl_info, ignore_banned=True)


Created sorted pl: PLWptjpDqazOwSXfulhbpYKmsqMf1Dc_t7, and  deleted original pl: PLWptjpDqazOxsy4e-ZhI8oH-k4Z8JAkTG


## Update Not Like tsv

In [7]:
not_like_name = os.path.splitext(os.path.basename(NOT_LIKE_PLAYLIST_TSV))[0]
not_like_pl = Y.playlists.loc[Y.playlists['title'] == not_like_name].iloc[0]
not_like_tracks, _ = Y.parse_playlist(Y.playlist_get_info(not_like_pl['playlistId'], use_cache=True))
not_like_tracks = not_like_tracks[PLAYLIST_TSV_COLUMNS].sort_values(['likeStatus', 'artist'], ascending=False)
not_like_tracks.to_csv(os.path.join(BACKUP_DIR, f'{not_like_name}.tsv'), sep='\t', header=True)

## Upload playlist from tsv backup

In [5]:
tsv = '../playlists/x_r.FunkSouMusic_tracks_like.tsv'
Y.playlist_from_tsv(tsv, ignore_banned=True, sort_by_index=True)


Generating x_r.FunkSouMusic_tracks_like ytmusic playlist for 49 tracks
Saved 49 x_r.FunkSouMusic_tracks_like tracks playlist with id: PLWptjpDqazOypETiPz5QXpvMDnJkm08jp


## Query

In [ ]:
Y.query_by_title(playlist_name)

## Get Playlist Counts

In [3]:
playlist_file = '../playlists/_playlist_radio_counts.tsv'
playlists = Y.get_playlist_counts(verbose=False, filter_title='radio')
radio_counts_df = playlists.loc[playlists.title.str.contains('radio')].sort_values('track_count')
radio_counts_df = radio_counts_df[['title', 'track_count', 'duration_hours', 'privacy', 'playlist_id']]
radio_counts_df.to_csv(playlist_file, sep='\t', index=False)
radio_counts_df
# last run 7-17-2023


,title,track_count,duration_hours,privacy,playlist_id
134,Hiphop southeast Ride Around Shining radio,17,1,PRIVATE,PLWptjpDqazOy0aKWVQk39xpseuXZ7s5AL
135,Indie Dreams of Fall radio,30,2,PRIVATE,PLWptjpDqazOyTZveTEz9fgHBmfL6suqTd
24,rock 1967 Monterey Pop Festival radio,33,2,PRIVATE,PLWptjpDqazOyoXjkoYrtOq-HxIFikPMb0
137,jazz essential radio,35,4,PRIVATE,PLWptjpDqazOxDKPeCgfbu_jp3Edn-Sh8M
139,jazz gloom smooth radio,38,4,PRIVATE,PLWptjpDqazOxD9ZM736lGLmWw5BQB-Z5D
...,...,...,...,...,...
59,x_r.2010smusic_tracks_radio,1106,72,PRIVATE,PLWptjpDqazOyU9d_yupbcHOq-9MYYOonW
75,x_r.futuregarage_tracks_radio,1142,97,PRIVATE,PLWptjpDqazOxj9ct9vG8tMeOMdj2rH_PQ
60,x_r.reggae_tracks_radio,1198,83,PRIVATE,PLWptjpDqazOxFpTJUsBAOp163DArs5Wyf
55,Reggae radio,1351,93,PRIVATE,PLWptjpDqazOwE761BnO1IfHJxwd9W8waZ


## Get Public Playlists

In [7]:
Y.get_playlists_by_privacy(privacy='PUBLIC')
# last run 6-2023, 3 public playlists

Found public playlist named: Liked Music
Found public playlist named: Chill Supermix
Found public playlist named: Episodes for Later


title                                                Liked Music
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
author                                                       NaN
title                                             Chill Supermix
playlistId           RDTMAK5uy_nzfwl2UYv7htL7wDoxbX8Pp6UAFBd92cQ
thumbnails     [{'url': 'https://music.youtube.com/image/mixa...
description                                        YouTube Music
count                                                        NaN
author                                                       NaN
title                                         Episodes for Later
playlistId                                                    SE
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description              

### Generate playlist froma a list of albums

In [ ]:
name = 'y_2022_albums_to_listen_to'
desc = 'manually selected albums to top off 2022 albums'
albums_to_add = [
    "SZA - SOS",
    "Perfume Genius - Ugly Season",
    "Arctic Monkeys - The Car",
    "The Beths - Expert in a Dying Field",
    "alt-J - The Dream",
     "Soichi Terada (寺田創一) - Asakusa Light",
    "Everything Everything - Raw Data Feel",
    "The Mountain Goats - Bleed Out",
    "Metric - Formentera",
    "Orville Peck - Bronco",
    "Hurray for the Riff Raff - Life on Earth",
    "Freddie Gibbs - $oul $old $eparately",
    "Tove Lo - Dirt Femme",
    "Oren Ambarchi - Shebang",
    "Hatchie - Giving the World Away",
    "Florence + the Machine - Dance Fever",
    "Richard Dawson - The Ruby Cord",
    "Charlotte Adigéry & Bolis Pupul - Topical Dancer",
    "Phoenix (FR) - Alpha Zulu",
    "Elder - Innate Passage",
    "The Black Angels - Wilderness of Mirrors",
    "Black Flower - Magma",
     "Bad Bunny - Un Verano Sin Ti",
     "Red Hot Chili Peppers - Return Of The Dream Canteen",

]

track_ids = []
for a in albums_to_add:
    match = {}
    res = Y.yt.search(query=a, filter='albums', limit=1)
    result_album = f"{res[0]['artists'][0]['name']} - {res[0]['title']}"
    if len(res) == 0 or res[0].get('browseId') == None:
        print(f'Skipping query: {a} bad result: {res}')
        continue
    yt_album = Y.yt.get_album(res[0]['browseId'])
    for t in yt_album['tracks']:
        track_ids.append(t['videoId'])
    print(f'Added {len(yt_album["tracks"])} tracks":\n\tq: {a}\n\tr: {result_album}')

pl_id = Y.yt.create_playlist( title=name,  description=desc, video_ids=track_ids)
print(f'Generated playlist: {name} with id {pl_id}')